# V-Max BC training on Colab

Runs `algorithm=bc` training from `as-fast-as-anyone` (V-Max fork) on a Colab GPU instead of the local GTX 1660 Super (6GB VRAM / 15GB RAM).

**Data source**: the contest organizer's raw 301-step TFRecord data (hanam/jeju folder layout), shared via Google Drive. Add that shared folder as a shortcut in your own Drive first (open the share -> "Add shortcut to Drive"), then set `RAW_301F_ROOT` below to wherever it lands.

This notebook re-runs the same local pipeline (`make_91f.py` -> `score_scenarios.py` -> `split_hard_easy_pools.py` -> `merge_pools.py`) on Colab to turn that raw data into the hard/easy BC pools, instead of trying to upload the already-built pools (those are symlinks into `train_91f` - a plain browser folder upload silently skips symlinks, so `data/shards/bc_pools/*` would upload as near-empty folders).

Also upload the small fixed 300-scenario **evaluation** set as a tar (so every model - local and Colab - is scored against the exact same set): locally, `tar -chf data/val_sample_shards_hanam.tar -C data/eval/val_sample_shards_hanam .` (~930MB, already dereferenced so no symlink issue), then upload that one file to `MyDrive/vmax_workdir/data/val_sample_shards_hanam.tar`.

Runtime > Change runtime type > select a GPU (T4 is free-tier; Colab Pro gives A100/L4).

**Why Drive at all (for runs/)**: Colab sessions disconnect (idle timeout / max runtime). Checkpoints are written straight to Drive (via a symlink), and BC training now supports full resume - if the session dies, just re-run the training cell with the same `name_run` and it picks up from the last checkpoint instead of restarting.

In [ ]:
!nvidia-smi

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Edit to wherever the contest organizer's shared-and-shortcut'd folder actually landed.
RAW_301F_ROOT = "/content/drive/MyDrive/<contest_shared_folder_name>"
DRIVE_WORKDIR = "/content/drive/MyDrive/vmax_workdir"

import os
assert os.path.isdir(RAW_301F_ROOT), f"Missing {RAW_301F_ROOT} - fix the path (did you Add shortcut to Drive?)."
print("Top-level contents:", os.listdir(RAW_301F_ROOT))
assert os.path.exists(f"{DRIVE_WORKDIR}/data/val_sample_shards_hanam.tar"), (
    f"Missing {DRIVE_WORKDIR}/data/val_sample_shards_hanam.tar - upload the fixed eval set tar first."
)

## 2. Clone the repo and set up the environment (uv, pinned by uv.lock)

In [ ]:
%cd /content
!rm -rf as-fast-as-anyone
!git clone https://github.com/gm2256/as-fast-as-anyone.git
%cd /content/as-fast-as-anyone/V-Max

!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"{os.path.expanduser('~')}/.local/bin:" + os.environ["PATH"]
!uv --version

In [ ]:
# Installs its own Python 3.12 (per .python-version) regardless of Colab's system Python,
# and resolves the exact versions pinned in uv.lock (same env as the local machine).
!uv sync

## 3. Smoke-test the conversion (30 files) before committing to the full run

Reading thousands of loose files off the Drive FUSE mount is slow and this pipeline touches every one of them at least twice (convert, then score) - worth 2 minutes to confirm the path/format is right before spending an hour-plus on the full dataset.

In [ ]:
!mkdir -p /content/data
!uv run python scripts/make_91f.py "$RAW_301F_ROOT" /content/data/train_91f_smoketest 30
!find /content/data/train_91f_smoketest -name '*.tfrecord' | wc -l

## 4. Full conversion + difficulty scoring + hard/easy split

Same 3 scripts as the local pipeline, same `--hard-frac 0.4` that produced the local pools. `make_91f.py` hardcodes 24 worker processes - fine even on Colab's 2 vCPUs (just oversubscribed, not broken), just don't expect a 12x speedup.

In [ ]:
!rm -rf /content/data/train_91f_smoketest
!uv run python scripts/make_91f.py "$RAW_301F_ROOT" /content/data/train_91f
!find /content/data/train_91f -name '*.tfrecord' | wc -l

In [ ]:
!mkdir -p /content/data/scores
!uv run python scripts/score_scenarios.py /content/data/train_91f /content/data/scores/combined_scores.csv 2

!uv run python scripts/split_hard_easy_pools.py \
    /content/data/scores/combined_scores.csv /content/data/train_91f /content/data/shards/mixture_pools --hard-frac 0.4
!uv run python scripts/merge_pools.py /content/data/shards/bc_pools \
    hard=/content/data/shards/mixture_pools/hanam_hard,/content/data/shards/mixture_pools/jeju_hard \
    easy=/content/data/shards/mixture_pools/hanam_easy,/content/data/shards/mixture_pools/jeju_easy

## 5. Wire up checkpoints (Drive, persistent) and the fixed eval set

In [ ]:
import os
os.makedirs(f"{DRIVE_WORKDIR}/runs", exist_ok=True)
!rm -rf /content/as-fast-as-anyone/V-Max/runs
!ln -s "$DRIVE_WORKDIR/runs" /content/as-fast-as-anyone/V-Max/runs

!mkdir -p /content/data/eval/val_sample_shards_hanam
!tar -xf "$DRIVE_WORKDIR/data/val_sample_shards_hanam.tar" -C /content/data/eval/val_sample_shards_hanam

## 6. Train

`@23570` / `@35355` should match the counts `merge_pools.py` printed in step 4 (same 178GB dataset, same `--hard-frac`, so should match the local pools) - edit if they differ.

`total_timesteps=5_000_000` is roughly one pass over the combined hard+easy pool (~59k scenarios x 80 steps). Bump it up (e.g. `20_000_000`, the framework's own default scale) once you've confirmed `train/imitation_loss` in TensorBoard is still trending down at 5M and want to keep going.

**If the session disconnects mid-run**: re-run steps 1-5 (data has to be rebuilt from Drive since Colab's local disk doesn't survive a disconnect), then this cell unchanged. `algorithm.resume=true` (default) picks up from `runs/<name_run>/model/train_state_latest.pkl` on Drive.

In [ ]:
%cd /content/as-fast-as-anyone/V-Max
!uv run python vmax/scripts/training/train.py \
  algorithm=bc network/encoder=lq \
  total_timesteps=5_000_000 num_envs=4 num_episode_per_epoch=1 \
  algorithm.buffer_size=20000 \
  waymo_dataset=true \
  'mixture_datasets=[{path: /content/data/shards/bc_pools/hard/hard.tfrecord@23570, weight: 0.3}, {path: /content/data/shards/bc_pools/easy/easy.tfrecord@35355, weight: 0.7}]' \
  name_run=colab_bc_run1 log_freq=50 save_freq=1500

## 7. Watch training in TensorBoard (optional, run in a separate cell while training runs)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/as-fast-as-anyone/V-Max/runs

## 8. After training: sweep checkpoints on the fixed held-out set

In [ ]:
%cd /content/as-fast-as-anyone/V-Max
!uv run python scripts/evaluate_checkpoints.py \
  --name_run colab_bc_run1 \
  --path_dataset /content/data/eval/val_sample_shards_hanam/val_sample_shards_hanam.tfrecord@300 \
  --waymo_dataset true --batch_size 4